# 03 · Classical pipeline improvements (Milestone 2b)

**SIBOL: System for Image-Based Observation and Learning**

Milestone 2 (notebook 02) established the classical baseline: from-scratch HOG + one-vs-rest linear SVM, **test top-1 0.1197, macro-F1 0.1024** (run 2). Its analysis showed the limit is the **features**, not the classifier: training accuracy ≈ 100 % in every SVM configuration, validation ≈ 14 %. HOG ignores colour, is a rigid spatial template, and the direct resize distorts aspect ratios.

This notebook tests four feature-side improvements, **one at a time, on the validation split only**:

| Step | Change | Why |
|---|---|---|
| 0 | HOG only (M2 baseline, re-scored with this notebook's SVM grid) | reference |
| 1 | **+ square-root normalisation** of the histogram features | histogram features usually suit a linear SVM better after a square root (Hellinger) |
| 2 | **+ HSV colour histograms** (hue × saturation, global + 2 × 2 grid, 320 values), colour weight 1 / 3 / 5 | HOG is grayscale; colour separates many flower classes |
| 3 | **+ horizontal-flip augmentation** of the training split (10 → 20 images per class) | more training data; flowers are left-right symmetric in meaning |
| 4 | **+ centre-crop resize** (largest centred square, aspect ratio kept) | the M2 direct resize stretches images (aspect 0.58–1.71) |

**Greedy forward selection:** each step is added to the best pipeline so far and **kept only if it raises validation macro-F1**. Only the final selected pipeline is evaluated on the **test split, once**. The M2 baseline remains the official Milestone 2 result; this is an extension (M2b), reported separately.

All features are written from scratch in NumPy (`src/classical/hog.py`, `color.py`). Flip augmentation is **exact and needs no re-extraction**: mirroring an image re-orders its HOG and colour features (proved by unit tests and checked on real images in Section 5).

### How to run in Colab
1. The M2b code must be pushed to GitHub (or copied to Drive and `CODE_DIR_COLAB` set).
2. **Runtime → Run all** on a **CPU runtime** (no GPU needed).
3. Expected time on the first run: about 5 min of new feature extraction (colour features, and HOG + colour for centre crop; the M2 HOG cache is reused), then about 15–20 min of SVM training. These are estimates from earlier Colab timings.

| What | Where (Colab) |
|---|---|
| Feature caches (HOG + colour, both resize modes) | `MyDrive/sibol/features/hog/` |
| Selected model, standardiser, recipe | `MyDrive/sibol/checkpoints/m2b_*` |
| Metrics (JSON/CSV) | `MyDrive/sibol/outputs/metrics/m2b_*` |
| Figures (PNG) | `MyDrive/sibol/outputs/figures/m2b_*` |

## 1. Environment setup

### 1.1 Configuration

In [ ]:
from pathlib import Path

# ---- SIBOL code (Colab) ------------------------------------------------------
REPO_URL = "https://github.com/didyouseejay/sibol.git"
CODE_DIR_COLAB = Path("/content/sibol")

# ---- Google Drive (Colab) ----------------------------------------------------
DRIVE_MOUNT_POINT = Path("/content/drive")
DRIVE_PROJECT_DIR = DRIVE_MOUNT_POINT / "MyDrive" / "sibol"
SOURCE_DIR_COLAB = DRIVE_PROJECT_DIR / "data"
FEATURES_DIR_COLAB = DRIVE_PROJECT_DIR / "features" / "hog"     # same cache folder as notebook 02
CHECKPOINTS_DIR_COLAB = DRIVE_PROJECT_DIR / "checkpoints"
METRICS_DIR_COLAB = DRIVE_PROJECT_DIR / "outputs" / "metrics"
FIGURES_DIR_COLAB = DRIVE_PROJECT_DIR / "outputs" / "figures"
WORK_ROOT_COLAB = Path("/content/flowers102")

# ---- Local runs (relative to the project root) ---------------------------------------
SOURCE_SUBDIR_LOCAL = "data"
WORK_SUBDIR_LOCAL = "data/extracted"
FEATURES_SUBDIR_LOCAL = "features/hog"
CHECKPOINTS_SUBDIR_LOCAL = "checkpoints"
METRICS_SUBDIR_LOCAL = "outputs/metrics"
FIGURES_SUBDIR_LOCAL = "outputs/figures"

# ---- Data behaviour ----------------------------------------------------------------
DOWNLOAD = False          # True = allow downloading MISSING source files from the official site
VERIFY_MD5 = True
FORCE_REEXTRACT = False

# ---- Features (HOG settings identical to Milestone 2) -------------------------------
IMAGE_SIZE = 128
CELL_SIZE, N_BINS, BLOCK_SIZE, BLOCK_STRIDE, L2HYS_CLIP = 8, 9, 2, 1, 0.2
COLOR_H_BINS, COLOR_S_BINS, COLOR_GRID = 16, 4, 2      # 5 regions x 16 x 4 = 320 values
RESIZE_MODES = ["direct", "center_crop"]

# ---- Ablation ----------------------------------------------------------------------
COLOR_WEIGHTS = [1.0, 3.0, 5.0]   # multiplier on the standardised colour columns (step 2)

# ---- Linear SVM grid used for EVERY recipe (6 configurations) -------------------------
# From M2 run 2: unweighted was best; lr 1e-3 was flat in lambda; lambda 10 helped lr 1e-2.
REG_GRID = [1e-4, 1.0, 10.0]
LR_GRID = [1e-3, 1e-2]
N_EPOCHS = 50
BATCH_SIZE = 128
LR_DECAY = 0.05
SEED = 42

# ---- Files --------------------------------------------------------------------------
PREFIX = "m2b"
M2_BASELINE_RESULTS = "hog_svm_run2_results.json"       # for the comparison table (read only)
M2_BASELINE_PER_CLASS = "hog_svm_run2_test_per_class.csv"
DEMO_INDEX = 0
N_FLIP_CHECK = 5

### 1.2 Runtime and Google Drive

In [ ]:
import importlib.util

IN_COLAB = importlib.util.find_spec("google.colab") is not None

if IN_COLAB:
    from google.colab import drive

    drive.mount(str(DRIVE_MOUNT_POINT))
    if not DRIVE_PROJECT_DIR.is_dir():
        raise FileNotFoundError(f"Google Drive is mounted, but {DRIVE_PROJECT_DIR} does not exist.")
    print(f"Google Drive mounted at {DRIVE_MOUNT_POINT}")
else:
    print("Running locally: Google Drive mount skipped.")

### 1.3 Load the SIBOL project code
Clones the repository in Colab if needed and refuses an older copy without the Milestone 2b modules.

In [ ]:
import subprocess
import sys

REQUIRED = [Path("src") / "data" / "flowers102.py", Path("src") / "classical" / "hog.py",
            Path("src") / "classical" / "color.py", Path("src") / "classical" / "experiment.py",
            Path("src") / "classical" / "features.py", Path("src") / "evaluation" / "metrics.py"]

if IN_COLAB:
    PROJECT_ROOT = CODE_DIR_COLAB
    if not (PROJECT_ROOT / REQUIRED[0]).is_file():
        if PROJECT_ROOT.exists() and any(PROJECT_ROOT.iterdir()):
            raise FileNotFoundError(f"{PROJECT_ROOT} exists but does not contain the SIBOL code. "
                                    f"Run !rm -rf {PROJECT_ROOT} and re-run this cell.")
        print(f"Cloning {REPO_URL} into {PROJECT_ROOT} ...")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_ROOT)], check=True)
else:
    PROJECT_ROOT = Path.cwd().resolve()
    while not (PROJECT_ROOT / REQUIRED[0]).is_file() and PROJECT_ROOT != PROJECT_ROOT.parent:
        PROJECT_ROOT = PROJECT_ROOT.parent

missing = [str(p) for p in REQUIRED if not (PROJECT_ROOT / p).is_file()]
if missing:
    raise FileNotFoundError(
        f"The SIBOL code in {PROJECT_ROOT} is missing Milestone 2b files: {missing}. It is probably an older "
        f"copy. In Colab: run !rm -rf {PROJECT_ROOT} and re-run this cell (push the M2b code first)."
    )

for name in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[name]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data import flowers102 as f102
from src.classical import color as colormod
from src.classical import experiment as E
from src.classical import features as feats
from src.classical import hog as hogmod
from src.classical.color import ColorConfig
from src.classical.hog import HOGConfig
from src.classical.linear_svm import SVMConfig
from src.classical.preprocessing import PreprocessConfig, preprocess_rgb, rgb_to_gray
from src.evaluation import metrics as M
from src.evaluation import plots as P

commit = subprocess.run(["git", "-C", str(PROJECT_ROOT), "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print(f"Project root : {PROJECT_ROOT}")
print(f"Git commit   : {commit or 'n/a (not a git checkout)'}")

### 1.4 Resolve paths and libraries

In [ ]:
import csv
import json
import time

import matplotlib.pyplot as plt
import numpy as np

try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = None

if IN_COLAB:
    SOURCE_DIR, WORK_ROOT = SOURCE_DIR_COLAB, WORK_ROOT_COLAB
    FEATURES_DIR, CHECKPOINTS_DIR = FEATURES_DIR_COLAB, CHECKPOINTS_DIR_COLAB
    METRICS_DIR, FIGURES_DIR = METRICS_DIR_COLAB, FIGURES_DIR_COLAB
else:
    SOURCE_DIR = PROJECT_ROOT / SOURCE_SUBDIR_LOCAL
    WORK_ROOT = PROJECT_ROOT / WORK_SUBDIR_LOCAL
    FEATURES_DIR = PROJECT_ROOT / FEATURES_SUBDIR_LOCAL
    CHECKPOINTS_DIR = PROJECT_ROOT / CHECKPOINTS_SUBDIR_LOCAL
    METRICS_DIR = PROJECT_ROOT / METRICS_SUBDIR_LOCAL
    FIGURES_DIR = PROJECT_ROOT / FIGURES_SUBDIR_LOCAL
for d in (FEATURES_DIR, CHECKPOINTS_DIR, METRICS_DIR, FIGURES_DIR):
    d.mkdir(parents=True, exist_ok=True)

ENV = {**M.environment_info(), "runtime": "Google Colab" if IN_COLAB else "Local", "git_commit": commit or None}
print(f"Runtime       : {ENV['runtime']}")
print(f"Feature cache : {FEATURES_DIR}")
print(f"Metrics       : {METRICS_DIR}")
print(f"Figures       : {FIGURES_DIR}")
print(f"DOWNLOAD      : {DOWNLOAD}")
print(f"NumPy {ENV['numpy']}, scikit-learn {ENV['scikit_learn']}, Python {ENV['python']}")

## 2. Dataset loading
Same steps as notebooks 01 and 02: verify the source files, extract to temporary storage, load the **official** splits, run the 10 integrity checks (the notebook stops if any fails).

In [ ]:
if DOWNLOAD:
    fetched = f102.download_source_files(SOURCE_DIR)
    print(f"Downloaded: {fetched or 'nothing (all files already present)'}")
else:
    missing = f102.missing_source_files(SOURCE_DIR)
    if missing:
        raise FileNotFoundError(f102.missing_files_message(SOURCE_DIR, missing))

source_report = f102.verify_source_files(SOURCE_DIR, check_md5=VERIFY_MD5)
for name, info in source_report.items():
    md5 = {True: "MD5 OK", False: "MD5 MISMATCH", None: "MD5 not checked"}[info["md5_ok"]]
    print(f"  {name:<16} {info['size']:>12,} bytes   {md5}")

reused = f102.is_prepared(WORK_ROOT)
t0 = time.time()
f102.prepare_dataset(SOURCE_DIR, WORK_ROOT, download=False, check_md5=False)
print("Reused existing extraction." if reused else f"Extracted images in {time.time() - t0:.0f} s.")

datasets = {split: f102.load_split(WORK_ROOT, split) for split in f102.SPLITS}
labels = {s: f102.get_labels(ds) for s, ds in datasets.items()}
files = {s: f102.get_image_files(ds) for s, ds in datasets.items()}
image_ids = {s: feats.image_ids_from_files(files[s]) for s in f102.SPLITS}

checks = f102.integrity_checks(labels, files)
for i, (name, passed, detail) in enumerate(checks, start=1):
    print(f"{i:>2}. [{'PASS' if passed else 'FAIL'}] {name}: {detail}")
n_passed = sum(passed for _, passed, _ in checks)
print(f"\n{n_passed}/10 checks passed.")
if n_passed != 10:
    raise RuntimeError("Integrity checks failed: investigate before using this data.")
CLASS_NAMES = getattr(datasets["train"], "classes", None) or [f"class_{i:03d}" for i in range(f102.NUM_CLASSES)]

## 3. Feature configuration

| Feature | Settings | Length |
|---|---|---:|
| HOG | identical to Milestone 2 (8 × 8 cells, 9 unsigned bins, 2 × 2 blocks, stride 1, L2-Hys) | 8,100 |
| Colour | HSV, 16 hue × 4 saturation bins; whole image + 2 × 2 grid; each region L1-normalised | 320 |

Both are computed for two resize modes: `direct` (M2) and `center_crop`. The `direct` HOG features are the Milestone 2 cache (same key), so they are not recomputed.

In [ ]:
HOG_CFG = HOGConfig(cell_size=CELL_SIZE, n_bins=N_BINS, block_size=BLOCK_SIZE, block_stride=BLOCK_STRIDE,
                    clip=L2HYS_CLIP)
COLOR_CFG = ColorConfig(h_bins=COLOR_H_BINS, s_bins=COLOR_S_BINS, grid=COLOR_GRID)
PRE = {mode: PreprocessConfig(image_size=IMAGE_SIZE, resize_mode=mode) for mode in RESIZE_MODES}
SPECS = {(kind, mode): (feats.hog_spec(PRE[mode], HOG_CFG) if kind == "hog" else feats.color_spec(PRE[mode], COLOR_CFG))
         for kind in ("hog", "color") for mode in RESIZE_MODES}
for (kind, mode), spec in SPECS.items():
    print(f"{kind:<6} {mode:<12} length {spec.feature_length:>5}  cache key {feats.spec_hash(spec)}")
assert SPECS[("hog", "direct")].feature_length == 8100 and COLOR_CFG.feature_length == 320
if feats.spec_hash(SPECS[("hog", "direct")]) != "b5525eb6a0c2":
    print("NOTE: HOG cache key differs from Milestone 2; the HOG features will be re-extracted.")

## 4. Feature extraction and caching
All feature sets are computed (or loaded from the validated cache) **before** any tuning. Each image's label from the loader is checked against the split's label list; caches are re-used only if configuration, split and the (image ID, label) fingerprint match.

In [ ]:
FEATS = {}   # (kind, mode) -> {split: FeatureSet}
for key, spec in SPECS.items():
    FEATS[key] = {}
    for split in f102.SPLITS:
        ds = datasets[split]
        fs, status = feats.get_or_extract(
            spec, split, image_ids[split], labels[split], ds.__getitem__, FEATURES_DIR, force=FORCE_REEXTRACT,
            progress=(lambda r, k=key, s=split: tqdm(r, desc=f"{k[0]}/{k[1]} {s}")) if tqdm else None)
        FEATS[key][split] = fs
        print(f"{key[0]:<6}{key[1]:<12}{split:<6} {fs.features.shape}  {status}  "
              f"({fs.meta['extraction_seconds_per_image'] * 1000:.1f} ms/image when extracted)")
    feats.assert_disjoint_splits(FEATS[key], expected_sizes=f102.EXPECTED_SPLIT_SIZES)
    assert all(np.isfinite(fs.features).all() for fs in FEATS[key].values())
    for split in f102.SPLITS:   # every feature set is aligned with the same split order
        assert FEATS[key][split].image_ids.tolist() == image_ids[split]
print("All feature sets: splits disjoint, official sizes, finite values, identical image order.")

## 5. Demonstrations and checks
1. The colour descriptor of one training image.
2. `direct` vs `center_crop` resizing of the same image.
3. **Flip check on real images:** features of the mirrored preprocessed image must equal the permuted features of the original (this is what makes flip augmentation exact).

In [ ]:
demo_image, demo_label = datasets["train"][DEMO_INDEX]
rgb_direct = preprocess_rgb(demo_image, PRE["direct"])
rgb_crop = preprocess_rgb(demo_image, PRE["center_crop"])
demo_hist = colormod.color_histogram(rgb_direct, COLOR_CFG)
P.plot_color_demo(rgb_direct, demo_hist, COLOR_H_BINS, COLOR_S_BINS, title=f"Colour descriptor: {CLASS_NAMES[demo_label]}",
                  path=FIGURES_DIR / f"{PREFIX}_color_demo.png")
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, img, title in zip(axes, [np.asarray(demo_image), rgb_direct, rgb_crop],
                          [f"original {demo_image.size[0]}x{demo_image.size[1]}", "direct (stretched)",
                           "center_crop (aspect kept)"]):
    ax.imshow(img)
    ax.set_title(title)
    ax.axis("off")
fig.tight_layout()
fig.savefig(FIGURES_DIR / f"{PREFIX}_resize_modes.png", dpi=150, bbox_inches="tight", facecolor="white")
plt.show()

flip_recipe = E.Recipe("check", color=True, flip=True)
perm = E.flip_index(flip_recipe, HOG_CFG, COLOR_CFG, IMAGE_SIZE)
max_diff = 0.0
for i in range(N_FLIP_CHECK):
    rgb = preprocess_rgb(datasets["train"][i][0], PRE["direct"])
    f = np.concatenate([hogmod.hog_features(rgb_to_gray(rgb), HOG_CFG), colormod.color_histogram(rgb, COLOR_CFG)])
    mirrored = rgb[:, ::-1]
    f_m = np.concatenate([hogmod.hog_features(rgb_to_gray(mirrored), HOG_CFG),
                          colormod.color_histogram(mirrored, COLOR_CFG)])
    max_diff = max(max_diff, float(np.abs(f_m - f[perm]).max()))
print(f"Flip check on {N_FLIP_CHECK} training images: max |features(mirror) - permuted features| = {max_diff:.2e}")
assert max_diff < 1e-9, "flip permutation does not reproduce mirrored-image features"

## 6. Ablation on the validation split (greedy forward selection)
For every recipe: build the inputs (square root → concatenate → flip-augment train → standardise on train only → weight colour columns), train the 6 SVM configurations on **train**, and score them on **validation**. A step is kept only if its best validation macro-F1 is **strictly higher** than the current pipeline's. The test split is not used in this section.

In [ ]:
SVM_CONFIGS = [SVMConfig(learning_rate=lr, reg_strength=reg, n_epochs=N_EPOCHS, batch_size=BATCH_SIZE,
                         lr_decay=LR_DECAY, positive_weight=None, seed=SEED)
               for lr in LR_GRID for reg in REG_GRID]

def label(r):
    parts = ["HOG"]
    if r.sqrt: parts.append("sqrt")
    if r.color: parts.append(f"colour(w={r.color_weight:g})")
    if r.flip: parts.append("flip")
    if r.resize_mode == "center_crop": parts.append("centre crop")
    return " + ".join(parts)

def run_recipe(r):
    hog = {s: FEATS[("hog", r.resize_mode)][s].features for s in ("train", "val")}
    col = {s: FEATS[("color", r.resize_mode)][s].features for s in ("train", "val")} if r.color else None
    design = E.build_design(r, hog, {s: np.asarray(labels[s]) for s in ("train", "val")}, col,
                            HOG_CFG, COLOR_CFG, IMAGE_SIZE)
    print(f"{label(r)}  [{design.X['train'].shape[0]} training rows x {design.n_features} features]")
    rows, best_row, model = E.svm_grid_search(design.X["train"], design.y["train"], design.X["val"], design.y["val"],
                                              SVM_CONFIGS, f102.NUM_CLASSES, M.classification_metrics, log=print)
    return {"recipe": r, "design": design, "rows": rows, "best_row": best_row, "model": model}

t_ablation = time.perf_counter()
current = E.Recipe("step 0")
current_res = run_recipe(current)
STEPS = [{"step": 0, "name": label(current), "recipe": current.to_dict(), "kept": True,
          "val_macro_f1": current_res["best_row"]["val_macro_f1"],
          "val_top1_accuracy": current_res["best_row"]["val_top1_accuracy"], "best_svm": current_res["best_row"]}]
ALL_GRID_ROWS = [{"recipe": label(current), **row} for row in current_res["rows"]]
print(f"  -> baseline val macro-F1 {STEPS[0]['val_macro_f1']:.4f}\n")

CANDIDATES = [
    (1, lambda c: [c.with_changes("step 1", sqrt=True)]),
    (2, lambda c: [c.with_changes("step 2", color=True, color_weight=w) for w in COLOR_WEIGHTS]),
    (3, lambda c: [c.with_changes("step 3", flip=True)]),
    (4, lambda c: [c.with_changes("step 4", resize_mode="center_crop")]),
]
for step, make in CANDIDATES:
    results = []
    for cand in make(current):
        res = run_recipe(cand)
        results.append(res)
        ALL_GRID_ROWS += [{"recipe": label(cand), **row} for row in res["rows"]]
    best = max(results, key=lambda r: (r["best_row"]["val_macro_f1"], r["best_row"]["val_top1_accuracy"]))
    kept = best["best_row"]["val_macro_f1"] > current_res["best_row"]["val_macro_f1"]
    for res in results:
        STEPS.append({"step": step, "name": label(res["recipe"]), "recipe": res["recipe"].to_dict(),
                      "kept": kept and res is best, "val_macro_f1": res["best_row"]["val_macro_f1"],
                      "val_top1_accuracy": res["best_row"]["val_top1_accuracy"], "best_svm": res["best_row"]})
    print(f"  -> step {step}: best {label(best['recipe'])} val macro-F1 {best['best_row']['val_macro_f1']:.4f} "
          f"vs current {current_res['best_row']['val_macro_f1']:.4f}: {'KEPT' if kept else 'not kept'}\n")
    if kept:
        current, current_res = best["recipe"], best
    else:
        del best
    del results
ABLATION_SECONDS = time.perf_counter() - t_ablation

SELECTED, SEL = current, current_res
print(f"Ablation took {ABLATION_SECONDS / 60:.1f} min. Selected pipeline: {label(SELECTED)}")
M.save_rows_csv([{k: v for k, v in s.items() if k not in ("recipe", "best_svm")} | {"svm_lr": s["best_svm"]["learning_rate"], "svm_reg": s["best_svm"]["reg_strength"]}
                 for s in STEPS], METRICS_DIR / f"{PREFIX}_ablation_steps.csv")
M.save_rows_csv(ALL_GRID_ROWS, METRICS_DIR / f"{PREFIX}_ablation_grid.csv")

## 7. Selected pipeline

In [ ]:
best_row, best_model, design = SEL["best_row"], SEL["model"], SEL["design"]
VAL_METRICS = M.classification_metrics(design.y["val"], best_model.predict(design.X["val"]), f102.NUM_CLASSES)
print(f"Pipeline   : {label(SELECTED)}")
print(f"SVM        : lr {best_row['learning_rate']:g}, lambda {best_row['reg_strength']:g}, {N_EPOCHS} epochs, unweighted")
print(f"Features   : {design.n_features:,}; training rows {design.X['train'].shape[0]:,}")
print(f"Parameters : {best_model.n_parameters:,} (W {best_model.W.shape[0]} x {best_model.W.shape[1]} + b {best_model.b.shape[0]})")
print("Validation :", {k: round(v, 4) for k, v in VAL_METRICS.items() if isinstance(v, float)})
print(f"Train acc  : {best_row['train_top1_accuracy']:.3f}")

best_model.save(CHECKPOINTS_DIR / f"{PREFIX}_best_svm.npz")
design.scaler.save(CHECKPOINTS_DIR / f"{PREFIX}_standardizer.npz")
M.save_json({"recipe": SELECTED.to_dict(), "label": label(SELECTED), "column_weights_colour": SELECTED.color_weight,
             "hog": HOG_CFG.to_dict(), "color": COLOR_CFG.to_dict(), "preprocess": PRE[SELECTED.resize_mode].to_dict()},
            CHECKPOINTS_DIR / f"{PREFIX}_recipe.json")
print(f"Saved model, standardiser and recipe to {CHECKPOINTS_DIR}")

## 8. Test evaluation (once)
The selected pipeline is applied to the test split exactly once. No augmentation is applied to test images.

In [ ]:
if globals().get("TEST_EVALUATED"):
    raise RuntimeError("The test split was already evaluated in this session. Restart the runtime to repeat the experiment.")

mode = SELECTED.resize_mode
test_hog = FEATS[("hog", mode)]["test"].features
test_col = FEATS[("color", mode)]["test"].features if SELECTED.color else None
y_test = np.asarray(labels["test"])

t0 = time.perf_counter()
X_test = E.transform_split(design, SELECTED, test_hog, test_col)
test_scores = best_model.decision_function(X_test)
test_pred = best_model.classes_[np.argmax(test_scores, axis=1)]
INFER_SECONDS = time.perf_counter() - t0
TEST_EVALUATED = True

assert test_scores.shape == (f102.EXPECTED_SPLIT_SIZES["test"], f102.NUM_CLASSES)
TEST_METRICS = M.classification_metrics(y_test, test_pred, f102.NUM_CLASSES)
PER_CLASS = M.per_class_metrics(y_test, test_pred, CLASS_NAMES)
CM = M.confusion(y_test, test_pred, f102.NUM_CLASSES)
print("Test:", {k: round(v, 4) for k, v in TEST_METRICS.items() if isinstance(v, float)})

extract_ms = {kind: FEATS[(kind, mode)]["test"].meta["extraction_seconds_per_image"] * 1000
              for kind in (["hog", "color"] if SELECTED.color else ["hog"])}
timing = {
    "feature_extraction_ms_per_image": extract_ms,
    "feature_extraction_note": "decode + preprocess + features, per feature type, measured when each cache was created; "
                               "each type decodes the image separately, so the sum is an upper bound",
    "svm_training_seconds_selected": best_row["fit_seconds"],
    "ablation_seconds_total": ABLATION_SECONDS,
    "inference_us_per_image": INFER_SECONDS / len(test_pred) * 1e6,
    "inference_note": "sqrt + standardise + weight + 102 scores + argmax on cached features",
}
RESULTS = {
    "milestone": "M2b: classical pipeline improvements (ablation)",
    "protocol": "greedy forward selection on validation macro-F1; selected pipeline evaluated once on test",
    "selected_pipeline": label(SELECTED), "selected_recipe": SELECTED.to_dict(),
    "selected_svm": best_model.config.to_dict(),
    "steps": [{k: v for k, v in s.items() if k != "best_svm"} | {"best_svm": {k: s["best_svm"][k] for k in
              ("learning_rate", "reg_strength", "train_top1_accuracy", "val_top1_accuracy", "val_macro_f1")}} for s in STEPS],
    "svm_grid": {"learning_rate": LR_GRID, "reg_strength": REG_GRID, "n_epochs": N_EPOCHS, "batch_size": BATCH_SIZE,
                 "lr_decay": LR_DECAY, "positive_weight": None, "seed": SEED},
    "features": {"hog": HOG_CFG.to_dict(), "color": COLOR_CFG.to_dict(), "n_features": design.n_features},
    "n_parameters": best_model.n_parameters,
    "validation": VAL_METRICS, "test": TEST_METRICS, "timing": timing, "environment": ENV,
    "created": time.strftime("%Y-%m-%d %H:%M:%S %Z"),
}
M.save_json(RESULTS, METRICS_DIR / f"{PREFIX}_results.json")
M.save_rows_csv(PER_CLASS, METRICS_DIR / f"{PREFIX}_test_per_class.csv")
M.save_matrix_csv(CM, METRICS_DIR / f"{PREFIX}_test_confusion_matrix.csv", CLASS_NAMES)
print(f"Results saved to {METRICS_DIR}")

## 9. Visualizations
Ablation chart, test confusion matrix, per-class F1, and the classes that changed most compared with the M2 baseline.

In [ ]:
P.plot_ablation(STEPS, title="M2b ablation: validation macro-F1 per recipe", path=FIGURES_DIR / f"{PREFIX}_ablation.png")
plt.show()
P.plot_confusion_matrix(CM, title=f"{label(SELECTED)}: test confusion matrix (row-normalised)",
                        path=FIGURES_DIR / f"{PREFIX}_confusion_matrix.png")
plt.show()
P.plot_per_class_f1(PER_CLASS, title=f"{label(SELECTED)}: per-class test F1", path=FIGURES_DIR / f"{PREFIX}_per_class_f1.png")
plt.show()

base_path = METRICS_DIR / M2_BASELINE_PER_CLASS
if base_path.is_file():
    with open(base_path, encoding="utf-8") as f:
        base_f1 = {int(r["class_id"]): float(r["f1"]) for r in csv.DictReader(f)}
    delta = sorted(((r["f1"] - base_f1[r["class_id"]], r["class_name"], base_f1[r["class_id"]], r["f1"]) for r in PER_CLASS),
                   reverse=True)
    print("Largest per-class test F1 gains vs M2 baseline:")
    for d, name, a, b in delta[:8]:
        print(f"  {name:<28} {a:.3f} -> {b:.3f} ({d:+.3f})")
    print("Largest losses:")
    for d, name, a, b in delta[-5:]:
        print(f"  {name:<28} {a:.3f} -> {b:.3f} ({d:+.3f})")
else:
    print(f"M2 per-class file not found at {base_path}; per-class comparison skipped.")

## 10. Results and observations

In [ ]:
print(f"Code commit       : {ENV['git_commit']}")
print(f"Selected pipeline : {label(SELECTED)}")
print("Ablation (validation macro-F1):")
for s in STEPS:
    print(f"  step {s['step']}  {s['name']:<45} {s['val_macro_f1']:.4f}  {'KEPT' if s['kept'] else ''}")
base_file = METRICS_DIR / M2_BASELINE_RESULTS
rows = [("val top1_accuracy", "validation", "top1_accuracy"), ("val macro_f1", "validation", "macro_f1"),
        ("test top1_accuracy", "test", "top1_accuracy"), ("test macro_precision", "test", "macro_precision"),
        ("test macro_recall", "test", "macro_recall"), ("test macro_f1", "test", "macro_f1")]
if base_file.is_file():
    base = json.loads(base_file.read_text(encoding="utf-8"))
    print(f"\n{'':<24}{'M2 (run 2)':>12}{'M2b':>12}")
    for name, part, k in rows:
        mine = VAL_METRICS[k] if part == "validation" else TEST_METRICS[k]
        print(f"{name:<24}{base[part][k]:>12.4f}{mine:>12.4f}")
else:
    print(f"\nM2 results not found at {base_file}; showing M2b only.")
    for name, part, k in rows:
        print(f"{name:<24}{(VAL_METRICS if part == 'validation' else TEST_METRICS)[k]:>12.4f}")
print(f"\nParameters {best_model.n_parameters:,}; inference {timing['inference_us_per_image']:.0f} µs/image; "
      f"ablation {ABLATION_SECONDS / 60:.1f} min")

### Observations

*To be written from the outputs above after the Colab run. Do not fill in numbers that were not produced by a run.* Points to cover:
- Which steps were kept, and by how much each changed validation macro-F1 (Section 6 and the ablation chart).
- Effect of colour, and which colour weight was best; which classes gained most (Section 9). Is the gain consistent with colour being the missing information?
- Train vs validation accuracy of the selected pipeline: is the over-fitting gap smaller?
- Validation vs test, and the comparison with the M2 baseline (Section 10).
- Cost: extra feature length, extraction time, training time with flip augmentation, parameters.
- Caveats: greedy selection on 1,020 validation images can keep small, noisy improvements; the test split was evaluated once in M2b (after twice in M2).